# Forecast robustness

Re-estimation (expanding and rolling window), evaluation without the 12–13 March event,
QLIKE loss, and Clark–West / Diebold–Mariano tests. All training rows satisfy
`target_end <= forecast time`. Exploratory: designed after viewing March results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

cwd = Path.cwd()
ROOT = cwd if (cwd / "requirements.txt").exists() else cwd.parent
if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Project root not found")

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

TARGET = "rv_next_1h_pct"
CONTROLS = ["rv_prev_1h_pct", "log_volume_prev_1h"]
SPECS = {
    "Controls": CONTROLS,
    "Controls_TR": CONTROLS + ["tr_vpin"],
    "Controls_U2": CONTROLS + ["u2_vpin"],
    "Controls_EX": CONTROLS + ["exchange_vpin"],
    "Controls_TR_EX": CONTROLS + ["tr_vpin", "exchange_vpin"],
}
ROLLING_HOURS = 720  # 30 days

train = pd.read_csv(PROCESSED / "forecast_train.csv")
test = pd.read_csv(PROCESSED / "forecast_test.csv")
exchange = pd.read_csv(PROCESSED / "btcusdt_exchange_vpin_2020_q1.csv")[
    ["bucket_id", "exchange_vpin"]
]

panel = pd.concat(
    [train.assign(sample="train"), test.assign(sample="test")], ignore_index=True
)
for column in ["timestamp", "target_end"]:
    panel[column] = pd.to_datetime(panel[column], utc=True)
panel = panel.sort_values("timestamp").reset_index(drop=True)
panel = panel.merge(exchange, on="bucket_id", how="left", validate="many_to_one")

assert panel["exchange_vpin"].notna().all()
assert panel["timestamp"].is_unique and panel["target_end"].is_monotonic_increasing

is_test = panel["sample"].eq("test").to_numpy()
test_pos = np.flatnonzero(is_test)
train_pos = np.flatnonzero(~is_test)
y_all = panel[TARGET].to_numpy(dtype=float)


def design(rows, features):
    return np.column_stack(
        [np.ones(len(rows)), panel.loc[rows, features].to_numpy(dtype=float)]
    )


def forecasts(scheme):
    """Forecasts for every test hour. Training rows always satisfy target_end <= forecast time."""
    out = {name: np.empty(len(test_pos)) for name in SPECS}

    if scheme == "fixed":
        for name, features in SPECS.items():
            beta = np.linalg.lstsq(
                design(train_pos, features), y_all[train_pos], rcond=None
            )[0]
            out[name][:] = design(test_pos, features) @ beta
        return out

    target_end = panel["target_end"].to_numpy()
    stamps = panel["timestamp"].to_numpy()
    for k, pos in enumerate(test_pos):
        n_avail = int(np.searchsorted(target_end, stamps[pos], side="right"))
        start = 0 if scheme == "expanding" else max(0, n_avail - ROLLING_HOURS)
        rows = np.arange(start, n_avail)
        assert panel["target_end"].iloc[rows].max() <= panel["timestamp"].iloc[pos]
        for name, features in SPECS.items():
            beta = np.linalg.lstsq(design(rows, features), y_all[rows], rcond=None)[0]
            out[name][k] = float((design(np.array([pos]), features) @ beta)[0])
    return out


ts = panel["timestamp"].iloc[test_pos].reset_index(drop=True)
te = panel["target_end"].iloc[test_pos].reset_index(drop=True)
event_start = pd.Timestamp("2020-03-12", tz="UTC")
event_end = pd.Timestamp("2020-03-14", tz="UTC")
SAMPLES = {
    "all_march": np.ones(len(ts), dtype=bool),
    "excluding_event": ((te <= event_start) | (ts >= event_end)).to_numpy(),
}
print("Event target hours:", int(((ts >= event_start) & (te <= event_end)).sum()))  # expect 48


def loss_table(y, f):
    e = f - y
    f_pos = np.maximum(f, 0.01)  # clipping only for QLIKE
    ratio = y**2 / f_pos**2
    return {
        "MSE": float(np.mean(e**2)),
        "MAE": float(np.mean(np.abs(e))),
        "bias": float(np.mean(e)),
        "QLIKE": float(np.mean(ratio - np.log(ratio) - 1)),
    }


def hac_test(d, maxlags):
    fit = sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    t = float(fit.tvalues[0])
    return (
        float(fit.params[0]),
        t,
        float(1 - stats.norm.cdf(t)),
        float(2 * (1 - stats.norm.cdf(abs(t)))),
    )


COMPARISONS = [  # (candidate, benchmark, test type)
    ("Controls_TR", "Controls", "CW"),
    ("Controls_U2", "Controls", "CW"),
    ("Controls_EX", "Controls", "CW"),
    ("Controls_TR_EX", "Controls_TR", "CW"),
    ("Controls_U2", "Controls_TR", "DM"),
    ("Persistence", "Controls_TR", "DM"),
]

metric_rows, test_rows = [], []

for scheme in ["fixed", "expanding", "rolling"]:
    pred = forecasts(scheme)
    pred["Persistence"] = panel["rv_prev_1h_pct"].to_numpy(dtype=float)[test_pos]
    y = y_all[test_pos]

    for sample, mask in SAMPLES.items():
        results = {m: loss_table(y[mask], f[mask]) for m, f in pred.items()}
        for model, res in results.items():
            metric_rows.append({
                "scheme": scheme, "sample": sample, "model": model, "n": int(mask.sum()),
                **res,
                "bias_sq": res["bias"] ** 2,
                "error_var": res["MSE"] - res["bias"] ** 2,
                "MSE_gain_vs_controls_pct": 100 * (1 - res["MSE"] / results["Controls"]["MSE"]),
                "MSE_gain_vs_persistence_pct": 100 * (1 - res["MSE"] / results["Persistence"]["MSE"]),
            })

        for candidate, benchmark, kind in COMPARISONS:
            e_c = pred[candidate][mask] - y[mask]
            e_b = pred[benchmark][mask] - y[mask]
            d = e_b**2 - e_c**2
            if kind == "CW":
                d = d + (pred[benchmark][mask] - pred[candidate][mask]) ** 2
            for lags in [6, 24]:
                mean_d, t_stat, p_one, p_two = hac_test(d, lags)
                test_rows.append({
                    "scheme": scheme, "sample": sample, "candidate": candidate,
                    "benchmark": benchmark, "test": kind, "hac_lags": lags,
                    "mean_loss_difference": mean_d, "t_stat": t_stat,
                    "p_one_sided": p_one, "p_two_sided": p_two,
                })

metrics = pd.DataFrame(metric_rows)
tests = pd.DataFrame(test_rows)

for scheme in ["fixed", "expanding", "rolling"]:
    for sample in SAMPLES:
        part = metrics[(metrics["scheme"] == scheme) & (metrics["sample"] == sample)]
        print(f"\n{scheme} | {sample}")
        print(part[["model", "n", "MSE", "MAE", "QLIKE", "bias",
                    "MSE_gain_vs_controls_pct"]].round(4).to_string(index=False))

print("\nTests (HAC lags = 6)")
print(tests[tests["hac_lags"] == 6].round(4).to_string(index=False))

metrics.to_csv(TABLES / "forecast_robustness_metrics.csv", index=False)
tests.to_csv(TABLES / "forecast_robustness_tests.csv", index=False)

Event target hours: 48

fixed | all_march
         model   n    MSE    MAE  QLIKE    bias  MSE_gain_vs_controls_pct
      Controls 740 1.7836 0.5766 2.0467 -0.5025                    0.0000
   Controls_TR 740 1.7709 0.5707 1.9603 -0.4971                    0.7131
   Controls_U2 740 1.7521 0.5623 1.8843 -0.4851                    1.7678
   Controls_EX 740 1.7839 0.5732 2.0723 -0.4945                   -0.0162
Controls_TR_EX 740 1.7633 0.5598 1.9542 -0.4766                    1.1384
   Persistence 740 1.4238 0.4872 1.0864  0.0003                   20.1747

fixed | excluding_event
         model   n    MSE    MAE  QLIKE    bias  MSE_gain_vs_controls_pct
      Controls 692 0.5324 0.4215 1.0031 -0.3430                    0.0000
   Controls_TR 692 0.5249 0.4160 0.9608 -0.3381                    1.4154
   Controls_U2 692 0.5117 0.4084 0.9137 -0.3268                    3.8843
   Controls_EX 692 0.5246 0.4174 0.9696 -0.3339                    1.4709
Controls_TR_EX 692 0.5039 0.4037 0.8695 -0.31